# Agents as tools (multi agent)

To have multiple agents work together, agents can be defined as tools that are provided to a manager agent.

In this example, an additional agent is responsible for evaluating the output and providing feedback to the manager agent.  Looking at the traces in OpenAI (https://platform.openai.com/logs?api=traces), the feedback agent provided feedback on transportaion and the result in the `trip_plan_using_multi_agents_ollama_and_serper.md` file contains additinal details realted to transportation between locations.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from instructions_multi_agent import TripPlannerInstructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


### The evaluation agent instructions

In [2]:
transportation_evaluation_agent_description = "A helpful assistant that evaluates the quality of trip plans based on user preferences and criteria."

transportation_evaluation_agent_instructions = """You are an expert trip planner evaluator. 
Your task is to assess the quality of trip plans, focusing on how clearly the transportation from location to location is described,
There should be clear instructions on how to get from place to place, including modes of transportation, estimated travel times, and any necessary transfers or connections.
This applies for all locations in the trip plan.
If the trip plan lacks clear transportation details, provide constructive feedback on how to improve it.
If the transportation details are clear and sufficient, respond with "The transportation details in the trip plan are clear and sufficient." """

In [3]:

# Generate custom instructions for the trip planner agent
planner = TripPlannerInstructions(
    output_file="trip_plan_using_multi_agents_ollama_and_serper.md"
)
custom_instructions = planner.get_instructions()
print(custom_instructions)
print("*" * 80)

# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

# Initialize the AsyncOpenAI client with the custom base_url
client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY,
)

# Specify the model you pulled with Ollama
# Note: Ollama expects just the model name (e.g., "llama3"), 
# not the full "gpt-oss" naming convention from the OpenAI API
#OLLAMA_MODEL_NAME = "gpt-oss:20b" 
OLLAMA_MODEL_NAME = "gpt-oss_131k_context:20b" 

# Wrap the client in the Agents SDK model class
model = OpenAIChatCompletionsModel(
    openai_client=client,
    model=OLLAMA_MODEL_NAME
)


sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
serper_params = {"command": "uvx", "args": ["serper-mcp-server"], "env": {"SERPER_API_KEY": os.environ.get('SERPER_API_KEY')}}

#web_search_tool = WebSearchTool(search_context_size="low")

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=45) as mcp_server_serper:
        transportation_evaluation_agent = Agent(
            model=model,
            name="transportation evaluation expert agent",
            instructions=transportation_evaluation_agent_instructions
        )

        trip_planner_agent = Agent(
            model=model,
            name="Trip Planner Agent",
            instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
            mcp_servers=[mcp_server_files, mcp_server_serper],
            tools=[
                transportation_evaluation_agent.as_tool(
                    tool_name="transportation_evaluation_expert",
                    tool_description=transportation_evaluation_agent_description,    
                )
            ]
        )
        with trace("Trip Planner Agent Ollama"):
            result = await Runner.run(trip_planner_agent, custom_instructions, max_turns=30)
            print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r